# Time Series

Time series data is a type of data that is collected and recorded at specific time intervals, typically in chronological order. Each observation in a time series corresponds to a particular point in time, and the key feature of this data is the temporal ordering—the order in which the data points occur matters and carries important information for analysis and forecasting.

Key features of time series data:

Temporal Dependence

Unlike typical datasets where observations are assumed to be independent, time series data often exhibits temporal dependence, meaning that past values influence future values. This sequential nature is central to how we model and forecast time series.

Regular vs. Irregular Intervals

Time series data can be:

Regularly spaced, e.g., hourly temperature readings, daily stock prices, monthly sales.
Irregularly spaced, e.g., timestamps of earthquakes or hospital admissions.
Most forecasting methods assume regular intervals, so irregular data may need preprocessing to become suitable for analysis.

Trends, Seasonality, and Cycles

Time series often contain patterns:

Trend: A long-term increase or decrease in the data.

Seasonality: Regular repeating patterns over fixed periods (e.g., daily, weekly, yearly).

Cycles: Fluctuations that are not of fixed period, often tied to economic or other external factors.

Understanding and separating these components is a major step in time-series modeling.


Stationarity

A time series is said to be stationary if its statistical properties—mean, variance, and autocorrelation—remain constant over time. Many modeling methods (like ARIMA) require the data to be stationary. We often transform non-stationary series using techniques like differencing.


Autocorrelation

Autocorrelation measures how related a current observation is to its past values. It helps determine lag relationships and is foundational in many time series models.


Time-series data uniqueness:

Order matters: You can't randomly shuffle rows—doing so destroys the time-based structure.
Past affects the future: The historical values can often explain or help predict future values.
Requires special methods: Standard ML techniques assume i.i.d. (independent and identically distributed) data, which doesn’t hold in time series. That’s why we use methods like ARIMA, exponential smoothing, and feature-based approaches adapted for time.

Parsing Dates and Setting the Time Index Time-series data typically comes with a timestamp column, which must be converted into a datetime format and set as the index of the DataFrame. This allows for powerful time-based operations such as resampling, shifting, and rolling calculations.

"""
import pandas as pd

df = pd.read_csv("daily_sales.csv")
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
"""

Visualizing Time-Series Data Plotting is the first step in understanding trends, seasonality, and potential outliers. Line plots help visualize the general shape of the series.

import matplotlib.pyplot as plt

df['sales'].plot(figsize=(12, 4), title="Daily Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

For seasonal patterns, you can use seasonal decomposition:

from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df['sales'], model='additive', period=7)
decomposition.plot()
plt.show()

Handling Missing Data Time-series datasets often have missing dates or values. One strategy is to fill in missing dates and impute missing values appropriately.

df = df.asfreq('D')  # Set frequency to daily
df['sales'].fillna(method='ffill', inplace=True)  # Forward fill

Resampling data

If your data is too granular or not aligned with the desired frequency, you can resample it. For example, converting daily data into monthly totals:

monthly_sales = df['sales'].resample('M').sum()
Smoothing and Noise Reduction

Smoothing techniques help highlight long-term trends by reducing short-term fluctuations. A common method is the moving average.

df['sales_smooth'] = df['sales'].rolling(window=7).mean()
df[['sales', 'sales_smooth']].plot(figsize=(12, 4))
This rolling average reduces day-to-day noise and makes trend patterns more visible.

Checking for Stationarity Many forecasting models, like ARIMA, assume the time series is stationary. You can visually check for stationarity by plotting the rolling mean and standard deviation, and statistically test it using the Augmented Dickey-Fuller (ADF) test.

from statsmodels.tsa.stattools import adfuller

result = adfuller(df['sales'].dropna())
print(f"ADF Statistic: {result[0]}")
print(f"p-value: {result[1]}")
A p-value less than 0.05 typically suggests the data is stationary. If it isn’t, differencing is often used to remove trends.

Differencing for Stationarity

Differencing subtracts the previous observation from the current one, removing linear trends.

df['sales_diff'] = df['sales'] - df['sales'].shift(1)
df['sales_diff'].dropna().plot(title='Differenced Sales')
Further differencing (e.g., seasonal) may be needed for seasonal trends.

Modelling

Once your data is prepared and explored, the next step is to apply forecasting models. Time-series models come in two broad categories: classical statistical models and machine learning approaches adapted for time series. Both have their strengths and can be used depending on the data characteristics and forecasting goals.

Naïve and Simple Models
Naïve forecasting is the simplest approach where the forecast for the next time step is just the last observed value. Despite its simplicity, it often serves as a strong baseline.

# Naïve forecast: predict next value as last observed
naive_forecast = df['sales'].shift(1)
Moving average smooths out short-term fluctuations by averaging over a fixed window:

df['moving_avg'] = df['sales'].rolling(window=3).mean()
These simple methods are useful to benchmark more complex models.

Exponential Smoothing Methods
Exponential smoothing models give more weight to recent observations, making them responsive to recent changes.

Simple Exponential Smoothing (SES): Suitable when data has no trend or seasonality.


from statsmodels.tsa.holtwinters import SimpleExpSmoothing

model = SimpleExpSmoothing(df['sales']).fit()
forecast = model.forecast(10)
Holt’s Linear Trend Method: Extends SES by modeling trends.

from statsmodels.tsa.holtwinters import Holt

model = Holt(df['sales']).fit()
forecast = model.forecast(10)
Holt-Winters Seasonal Method: Handles both trend and seasonality, with additive or multiplicative components.
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(df['sales'], trend='add', seasonal='add',
														 seasonal_periods=12).fit()
forecast = model.forecast(12)
Exponential smoothing models are fast, interpretable, and perform well on many datasets.

ARIMA Models

ARIMA (AutoRegressive Integrated Moving Average) is a very popular family of models that combine autoregression (AR), differencing (I for integration), and moving average (MA) components to handle different types of non-stationary data.

AR(p): Uses lagged values of the series.
I(d): Number of times data is differenced to achieve stationarity.
MA(q): Uses lagged forecast errors.

from statsmodels.tsa.arima.model import ARIMA

# Fit ARIMA model with parameters (p=2, d=1, q=2)
model = ARIMA(df['sales'], order=(2, 1, 2))
model_fit = model.fit()
forecast = model_fit.forecast(steps=10)
Selecting the best parameters often involves analyzing the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) plots, along with information criteria like AIC or BIC.

Seasonal ARIMA (SARIMA)

SARIMA extends ARIMA to explicitly model seasonal effects by adding seasonal autoregressive, differencing, and moving average terms.

The model order is specified as ARIMA(p,d,q)(P,D,Q)[s], where [s] is the seasonal period (e.g., 12 for monthly data with yearly seasonality).


from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(df['sales'], order=(1,1,1), seasonal_order=(1,1,1,12))
model_fit = model.fit()
forecast = model_fit.forecast(steps=12)
SARIMA is well-suited for data with strong seasonal patterns.

Machine Learning Approaches for Time Series

Machine learning models don’t explicitly model time dependencies but can be adapted by framing forecasting as a supervised learning problem.

Key steps:

Create lagged features and rolling statistics.
Include time-based features (day of week, month).
Split data chronologically to prevent data leakage.
Example of creating lag features:

df['lag_1'] = df['sales'].shift(1)
df['lag_7'] = df['sales'].shift(7)
df['rolling_mean_7'] = df['sales'].rolling(window=7).mean()
df.dropna(inplace=True)
You can then train models like:

Linear Regression
Random Forest
Gradient Boosting Machines (XGBoost, LightGBM)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

features = ['lag_1', 'lag_7', 'rolling_mean_7']
X = df[features]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False,
test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
Machine learning models can capture complex nonlinear relationships and incorporate external variables but require careful feature engineering and validation.

Evaluation

Evaluating the performance of your forecasting models is a crucial step to ensure they make reliable predictions on unseen data. Unlike traditional machine learning, time-series forecasting requires special attention to the temporal order of data, meaning you cannot simply shuffle and randomly split your dataset. Instead, evaluation strategies must respect the sequence of observations to avoid data leakage and overly optimistic results.

The most straightforward method is to split your time-series data into training and testing sets based on time. For example, you might train on the first few years of data and test on the most recent months.

However, to get a more robust estimate of model performance, you can use:

Walk-Forward Validation (Rolling Forecast Origin): The model is trained on an initial window, then tested on the next period. After that, the training window moves forward, and the process repeats. This mimics real forecasting scenarios where new data continuously arrives.
Expanding Window Validation: The training set grows with each step while the test set moves forward in time.
Both methods allow you to measure how well your model adapts as more data becomes available.

Since you’re familiar with regression metrics such as: MAE, MSE and RMSE, here are some additional metrics useful for time series:

Mean Absolute Percentage Error (MAPE): Expresses errors as a percentage, making it easier to interpret across different scales. Keep in mind it can be unstable when actual values are close to zero. https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html
Symmetric Mean Absolute Percentage Error (sMAPE): A variant of MAPE that balances the relative error by considering both actual and predicted values, reducing issues near zero. https://epftoolbox.readthedocs.io/en/latest/modules/metrics/smape.html
Forecast Bias: Measures whether your model consistently over- or under-predicts by averaging the signed errors. A value close to zero means your model is unbiased. https://machinelearningmastery.com/time-series-forecasting-performance-measures-with-python/
Prediction Intervals: Evaluating the uncertainty of forecasts can be crucial in many applications. Methods like ARIMA or exponential smoothing can provide prediction intervals, which tell you the range where future values are likely to fall. https://www.geeksforgeeks.org/data-analysis/confidence-and-prediction-intervals-with-statsmodels/